# 00 · Setup (run first in every Colab session)

This notebook mounts Google Drive, clones or updates the project repo, installs it, checks the GPU, creates the project folders on Drive and runs the unit tests.

Before running:
1. **Runtime → Change runtime type → T4 GPU.**
2. Set `REPO_URL` below to your GitHub repo. For a private repo, use a GitHub personal access token through Colab's **Secrets** panel (key icon) rather than pasting it into the notebook.

If pip changes an already-imported package such as numpy, Colab asks you to restart the runtime. Restart, then run the notebook again; the second run is a no-op install.

In [1]:
!nvidia-smi

Wed Oct 28 18:13:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

# ---- Edit these ----
REPO_URL = "https://github.com/<github-user>/<repo-name>.git"
BRANCH = "main"
# --------------------

REPO_DIR = "/content/cs576-change-detection"
PROJECT_ROOT = "/content/drive/MyDrive/cs576-change-detection"  # data, runs, results

# Every script and notebook reads the project root from this variable.
os.environ["CD_PROJECT_ROOT"] = PROJECT_ROOT

In [ ]:
# Clone on the first run of a session, pull afterwards.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
# Pinned extras first, then the package itself without touching dependencies,
# so Colab's preinstalled torch/torchvision are never replaced.
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

In [ ]:
import sys

# The editable install takes effect on the next kernel start; this makes `cd`
# importable in the current session too.
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)

import numpy as np
import torch

import cd
from cd.utils import get_device, get_paths, set_seed

print(f"cd {cd.__version__} | torch {torch.__version__} | numpy {np.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", get_device())

set_seed(42)
paths = get_paths().make_dirs()
for name in ("root", "raw", "tiles", "stats_file", "runs", "results"):
    print(f"{name:>10}: {getattr(paths, name)}")

In [ ]:
!python -m pytest -q

## Done

You should see the T4 in `nvidia-smi`, `CUDA available: True`, the Drive folders listed above, and all tests passing.

Next: put the LEVIR-CD download in `data/raw` on Drive (Task 3 tiles it).